# 🇻🇳 ViMind 4.0: Bước Nhảy Nhận Thức (The Cognitive Sovereign)
### (Mixture-of-Experts 198M Top-2 Routing, Targeted Pre-training, CoT SFT, DPO & Agent RL)

Notebook này thiết lập chu trình khép kín tối tân nhất cho **ViMind 4.0** trên **Kaggle GPU (Tesla T4 16GB VRAM)**:
1. **Pha 1: Targeted Pre-training (MoE 198M - Top-2 Routing):** Tiền huấn luyện trên Wikipedia tiếng Việt, Sách giáo khoa Toán/Khoa học, và Tri thức nền tảng Việt Nam. Xây dựng bản sắc và vốn tri thức thực tế trước khi SFT.
2. **Pha 2: SFT 4.0 (CoT & Factual Grounding):** Nạp từ Base Model vừa tiền huấn luyện, học 52,000+ mẫu hội thoại tiếng Việt kết hợp chuỗi suy nghĩ `<think>` và Factual Grounding.
3. **Pha 3: Anti-Refusal DPO:** Căn chỉnh sở thích, loại bỏ hoàn toàn hiện tượng từ chối máy móc và ảo giác.
4. **Pha 4: Agentic RL (GRPO Tool-Use):** Huấn luyện kỹ năng kích hoạt công cụ giải toán, tra thời tiết, thời gian qua `<tool_call>`.
5. **Pha 5: Anti-Degeneration Inference & Packaging:** Đóng gói SafeTensors và kiểm thử suy luận với Repetition Penalty 1.2 & N-gram blocking.

In [ ]:
# 1. Đồng bộ mã nguồn ViMind 4.0 từ GitHub
!rm -rf /kaggle/working/vimind
!git clone https://github.com/WuKong0601/ViMind.git /kaggle/working/vimind
%cd /kaggle/working/vimind


In [ ]:
# 2. Cài đặt các gói phụ thuộc (PyTorch, Transformers, BitsAndBytes, SafeTensors, FastAPI)
!pip install -r requirements.txt
!pip install -q bitsandbytes accelerate fastapi uvicorn requests


In [ ]:
# 3. Kiểm tra phần cứng GPU Tesla T4 & môi trường CUDA
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {gpu_name} ({total_mem:.1f} GB VRAM)')


In [ ]:
# 4. Chạy Dry-Run kiểm thử toàn diện 5 thành phần ViMind 4.0
# (Kiểm tra MoE Top-2 Routing, Anti-Degeneration Decoding, Transformers 5.0+ Compatibility)
!python trainer/test_4.0_dry_run.py


In [ ]:
# 5. [GIAI ĐOẠN 1A: CHUẨN BỊ DỮ LIỆU TIỀN HUẤN LUYỆN & SFT 4.0]
# Tự động tổng hợp Wikipedia tiếng Việt, Sách giáo khoa Toán, và Factual Grounding
import os
if not os.path.exists('dataset/pretrain_vi_v4.jsonl') or os.path.getsize('dataset/pretrain_vi_v4.jsonl') < 100000:
    print('📚 Biên dịch tập dữ liệu tiền huấn luyện chất lượng cao...')
    !python data_pipeline/prepare_pretrain_4.0.py --max_wiki 60000 --max_math 20000
else:
    print('⚡ Đã có sẵn dataset/pretrain_vi_v4.jsonl!')

if not os.path.exists('dataset/sft_vi_v4.jsonl'):
    print('🎯 Biên dịch tập dữ liệu SFT 4.0 (52k + Grounding Oversampling)...')
    !python data_pipeline/download_distilled_4.0.py
else:
    print('⚡ Đã có sẵn dataset/sft_vi_v4.jsonl!')


In [ ]:
# 6. [GIAI ĐOẠN 1B: TARGETED PRE-TRAINING (MoE 198M - Top-2 Routing)]
# Tiền huấn luyện mạng nơ-ron từ tri thức bách khoa, loại bỏ vĩnh viễn việc train từ số 0
!python -u trainer/pretrain.py \
    --data_path dataset/pretrain_vi_v4.jsonl \
    --tokenizer_dir model \
    --save_dir out/pretrain \
    --save_weight vimind_4.0_base \
    --model_size 64m \
    --use_moe \
    --num_experts 4 \
    --num_experts_per_tok 2 \
    --batch_size 16 \
    --accumulation_steps 8 \
    --epochs 1 \
    --learning_rate 4e-4 \
    --dtype float16 \
    --log_interval 50 \
    --save_interval 500


In [ ]:
# 7. [GIAI ĐOẠN 2: HUẤN LUYỆN SFT 4.0 (Nạp trực tiếp từ Base Model đã có tri thức)]
# Huấn luyện 2 epochs với MoE 4 experts và Top-2 Gating
base_checkpoint = 'out/pretrain/vimind_4.0_base_final' if os.path.exists('out/pretrain/vimind_4.0_base_final') else 'none'
!python -u trainer/train_sft.py \
    --data_path dataset/sft_vi_v4.jsonl \
    --from_pretrained {base_checkpoint} \
    --tokenizer_dir model \
    --save_dir out/sft_moe \
    --save_weight vimind_4.0_moe \
    --use_moe \
    --num_experts 4 \
    --num_experts_per_tok 2 \
    --batch_size 16 \
    --accumulation_steps 4 \
    --epochs 2 \
    --learning_rate 1.5e-4 \
    --dtype float16 \
    --log_interval 25 \
    --save_interval 500


In [ ]:
# 8. [GIAI ĐOẠN 3: DIRECT PREFERENCE OPTIMIZATION (DPO 4.0)]
# Căn chỉnh mô hình theo nhãn chất lượng sạch (khử hoàn toàn từ chối máy móc và tiêm factual grounding)
import os
!python data_pipeline/filter_dpo_refusals.py --input_path dataset/dpo_qwen_judged.jsonl --output_path dataset/dpo_qwen_judged.jsonl
dpo_data = 'dataset/dpo_qwen_judged.jsonl' if os.path.exists('dataset/dpo_qwen_judged.jsonl') else 'dataset/dpo_vi.jsonl'
!python -u trainer/train_dpo.py \
    --data_path {dpo_data} \
    --model_path out/sft_moe \
    --save_dir out/dpo \
    --save_weight vimind_4.0_dpo \
    --epochs 1 \
    --batch_size 4 \
    --accumulation_steps 4 \
    --learning_rate 1e-5 \
    --fp16


In [ ]:
# 9. [GIAI ĐOẠN 4: AGENTIC REINFORCEMENT LEARNING (GRPO 4.0)]
# Huấn luyện khả năng sử dụng công cụ Toán học, Thời tiết, Thời gian và suy nghĩ logic
import os
base_rl_model = 'out/dpo' if os.path.exists('out/dpo') else 'out/sft_moe'
!python -u trainer/train_agent.py \
    --model_path {base_rl_model} \
    --save_dir out/agent_rl \
    --save_weight vimind_4.0_agent \
    --epochs 1 \
    --num_rollouts 4 \
    --learning_rate 1e-5 \
    --fp16


In [ ]:
# 10. [GIAI ĐOẠN 5: BENCHMARK ĐÁNH GIÁ NĂNG LỰC GỌI CÔNG CỤ (TOOL CALL)]
bench_model = 'out/agent_rl' if os.path.exists('out/agent_rl') else ('out/dpo' if os.path.exists('out/dpo') else 'out/sft_moe')
!python scripts/eval_toolcall.py --model_path {bench_model}


In [ ]:
# 11. [GIAI ĐOẠN 6: ĐÓNG GÓI & XUẤT XƯỞNG HUGGING FACE SAFETENSORS 4.0]
# Chuyển đổi trọng số sang chuẩn SafeTensors, tích hợp Chat Template và Tokenizer 4.0
import os
source_weight = 'out/agent_rl/vimind_4.0_agent.pth' if os.path.exists('out/agent_rl/vimind_4.0_agent.pth') else ('out/dpo/vimind_4.0_dpo.pth' if os.path.exists('out/dpo/vimind_4.0_dpo.pth') else 'out/sft_moe/vimind_4.0_moe.pth')
!python scripts/convert_model.py \
    --input {source_weight} \
    --output /kaggle/working/vimind_4.0_moe_final \
    --moe \
    --num_experts 4


In [ ]:
# 12. [GIAI ĐOẠN 7: KIỂM THỬ THÀNH PHẨM VIMIND 4.0 VỚI ANTI-DEGENERATION INFERENCE]
# Thử nghiệm: Factual Grounding (Hà Nội, 63 tỉnh thành), Phép tính 600, Lý luận <think>, và Gọi công cụ <tool_call>
import os, json, torch
from transformers import AutoTokenizer
from model.model import ViMindConfig, ViMindForCausalLM

model_dir = '/kaggle/working/vimind_4.0_moe_final'
tok_dir = model_dir if os.path.exists(os.path.join(model_dir, 'tokenizer.json')) else 'model'
print(f'📦 Nạp Tokenizer từ: {tok_dir}')
tokenizer = AutoTokenizer.from_pretrained(tok_dir)

if os.path.exists(model_dir):
    config = ViMindConfig.from_pretrained(model_dir) if os.path.exists(os.path.join(model_dir, 'config.json')) else ViMindConfig(use_moe=True, num_experts=4, num_experts_per_tok=2)
    model = ViMindForCausalLM(config).cuda()
    from safetensors.torch import load_file
    sf_path = os.path.join(model_dir, 'model.safetensors')
    if os.path.exists(sf_path):
        model.load_state_dict(load_file(sf_path), strict=False)
        print('✅ Đã nạp thành công trọng số safetensors!')
    model.eval()

    test_cases = [
        'Xin chào, bạn là mô hình AI nào và bạn có khả năng gì nổi bật?',
        'Thủ đô của nước Cộng hòa Xã hội Chủ nghĩa Việt Nam là gì?',
        'Việt Nam có bao nhiêu tỉnh thành? Hãy kể tên một số tỉnh miền Trung.',
        'Tính giúp tôi kết quả của 25 * 18 + 750 / 5.',
        'Thời tiết hôm nay tại Đà Nẵng thế nào, có mát mẻ không?',
        'Vì sao ban ngày trời sáng còn ban đêm trời tối?'
    ]

    print('=' * 70)
    print('      🎉 KẾT QUẢ ĐỐI THOẠI TRỰC TIẾP VỚI VIMIND 4.0 MOE')
    print('=' * 70)
    for q in test_cases:
        try:
            prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt, return_tensors='pt').input_ids.cuda()
            with torch.no_grad():
                out = model.generate(
                    inputs,
                    max_new_tokens=256,
                    temperature=0.7,
                    top_p=0.85,
                    repetition_penalty=1.2,
                    no_repeat_ngram_size=3,
                    eos_token_id=tokenizer.eos_token_id
                )
            reply = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False)
            print(f'\n👤 Người dùng: {q}')
            print(f'🤖 ViMind 4.0:\n{reply.strip()}')
            print('-' * 70)
        except Exception as e:
            print(f'⚠️ Lỗi test câu hỏi "{q}": {e}')
